# Factores de Certidumbre

## Introducción

La Inteligencia Artificial (IA) ha sido tradicionalmente dividida en dos principales paradigmas:
* **IA simbólica**: basada en representaciones explícitas del conocimiento (símbolos, lógica, reglas)
* **IA subsimbólica**: también conocida como conexionista, se basa en métodos numéricos para simular conexiones neuronales (aprendizaje máquina)

Un reto clave para la IA simbólica es el **razonamiento con conocimiento inexacto**. En problemas del mundo real, es posible y probable por varios motivos:

* Información incompleta o faltanta
* Errores o imprecisiones al medir
* Incertidumbre en las relaciones causales
* Desacuerdo entre expertos
* Entornos no-deterministas

A lo largo de la literatura, se han propuesto varios modelos para tratar con estas cuestiones. Uno de los primeros y de los más influyentes es el modelo de **factores de certidumbre (CF)**, introducido por Shortliffe y Buchanan.

Aunque el modelo no se basa formalmente en la teoría de probabilidad, resulta **intuitivo, eficiente y efectivo en la práctica**, al proporcionar herramientas para representar y combinar conocimiento inexacto en sistemas basados en reglas.

## Modelo

El modelo de factores de certidumbre considera hipótesis *h* y evidencias *e*, a partir de los cuales define dos índices:

* MB(h,e): *measure of belief*,  medida del incremento en la creencia de *h* dada *e*
* MD(h,e): *measure of disbelief*,  medida del decremento en la creencia de *h* dada *e*

Estas medidas **no son absolutas**, si no que representan cambios en la creencia causados por nuevas evidencias. 

Partiendo de una hipótesis *h* y una evidencia *e*, se tiene que:

* $p(h)$: probabilidad a priori de $h$
* $p(h \mid e)$: probabilidad a posteriori de $h$ tras observar la evidencia $e$

### Índices de creencia

En función de estos valores, se dan **tres casos**:

Si:

$$p(h \mid e) > p(h)$$

Entonces:

$$MB(h, e) = \frac{p(h \mid e) - p(h)}{1 - p(h)}, \quad MD(h, e) = 0$$

Esto representa un **incremento relativo de la creencia**, ya que la evidencia apoya la hipótesis.

---

Si:

$$p(h \mid e) < p(h)$$

Entonces:

$$MD(h, e) = \frac{p(h) - p(h \mid e)}{p(h)}, \quad MB(h, e) = 0$$

Esto representa un **decremento relativo de la creencia**, ya que la evidencia no apoya la hipótesis.

---

Si:

$$p(h \mid e) = p(h)$$

Entonces:

$$MB(h, e) = MD(h, e) = 0$$

Esto representa una **independencia de la evidencia**, ya que no influye sobre la hipótesis (o no se tiene el conocimiento suficiente como para establecer una relación).

---

Ambos índices se encuentran siempre en el rango $[0,1]$, debido a su relación con las probabilidades que los definen.

#### Ejercicio 1

Implementa los índices MB y MD en Python. Utiliza el diccionario de datos para comprobar los resultados.

In [ ]:
knowledge_base = {
    'p(h)': 0.5,
    'p(h|e)': 0.8,
    # ...
}

def mb_md_indexes(h,e):
    """Function that calculates the MB(h,e) and MD(h,e) indexes.

    Args:
        h (float): Value of p(h).
        e (float): Value of p(h|e).
        
    Returns:
        ((float,float)): Tuple with the indexes (MB,MD)
    """
    
    mb = 0
    md = 0
    
    # YOUR CODE HERE
    
    return (mb, md)

print('Expected: ', (0.6, 0))
print('Obtained: ', mb_md_indexes(knowledge_base.get('p(h)'),knowledge_base.get('p(h|e)')))

### Factor de certidumbre (CF)

Además de los índices de creencia, se define un tercer índice que los agrega, para facilitar la evaluación y comparación:

$$CF(h, e) = MB(h, e) - MD(h, e)$$

Dada la definición de los índices de creencia, el factor de certidumbre se encuentra en el rango $[-1,1]$, saliéndose de los límites de la probabilidad clásica.

El objetivo de definir estos conceptos viene de la idea de Shortliffe y Buchanan de que en la teoría probabilística, la misma evidencia apoya la creencia en sentidos positivo y negativo:

$$p(h|e) + p(\lnot h|e) = 1$$

Sin embargo, el nuevo coeficiente $CF$ sostiene que:

$$CF(h,e) = -CF(\lnot h,e)$$

Esto implica que los factores de certidumbre **no son complementarios como las probabilidades**, lo que los distingue del enfoque probabilístico clásico.

### Combinación de evidencias

Sea un conjunto de evidencias $E = \{e_1, e_2, ..., e_n\}$ que apoyan una misma hipótesis:

$$
\begin{align*}
    e_1 \xrightarrow[]{CF(e_1,h)} h \\
    e_2 \xrightarrow[]{CF(e_2,h)} h \\
    \cdots \hspace{2em} \\
    e_n \xrightarrow[]{CF(e_n,h)} h
\end{align*}
$$

Queremos calcular:

$$
CF(h, E)
$$

Existen **reglas de combinación** para poder agrupar los distintos CFs por pares de forma recursiva:

---

Si ambas son positivas $(CF(h, e_1) > 0$ y $CF(h, e_2) > 0)$:

$$
CF(h, e_1 \land e_2) = CF(h, e_1) + CF(h, e_2) - CF(h, e_1)\cdot CF(h, e_2)
$$

---

Si ambas son negativas $(CF(h, e_1) < 0$ y $CF(h, e_2) < 0)$:

$$
CF(h, e_1 \land e_2) = CF(h, e_1) + CF(h, e_2) + CF(h, e_1)\cdot CF(h, e_2)
$$

---

Si son de signo diferente $(CF(h, e_1) * CF(h, e_2) < 0)$:

$$
CF(h, e_1 \land e_2) =
\frac{CF(h, e_1) + CF(h, e_2)}
{1 - \min\left(|CF(h, e_1)|, |CF(h, e_2)|\right)}
$$

---

Todas las operaciones son **asociativas**, por lo que el orden de combinación no afecta al resultado final.

#### Ejercicio 2

Implementa la función para el factor de certidumbre a partir de los índices de creencia y las reglas de combinación.

In [ ]:
knowledge_base = {
    'p(h)': 0.5,
    'p(h|e1)': 0.8,
    'p(h|e2)': 0.2,
    'p(h|e3)': 0.3,
    # ...
}

def cf(h,e):
    """Function that calculates the certainty factor CF(h,e)

    Args:
        h (float): Value of p(h).
        e (float): Value of p(h|e).
        
    Returns:
        (float): Certainty factor
    """
    
    ### YOUR CODE HERE
    
    pass

def cf_combined(h,e1,e2):
    """Function that combines the certainty factors CF(h,e1) and CF(h,e2)

    Args:
        h (float): Value of p(h).
        e1 (float): Value of p(h|e1).
        e2 (float): Value of p(h|e2).
        
    Returns:
        (float): Combined certainty factor
    """
    
    ### YOUR CODE HERE
    
    pass

### Propagación de la inexactitud

El encadenamiento de reglas es una práctica común en el desarrollo de sistemas de reglas:

$$e_1 \xrightarrow[]{CF(e_1,e_2)} e_2 \xrightarrow[]{CF(e_2,h)} h$$

Según el modelo de factores de certidumbre, esta relación se aplica como sigue:

$$CF(h, e_1) = CF(h, e_2) \cdot \max(0, CF(e_1, e_2))$$

Este planteamiento se aprovecha para modelar la imprecisión de los datos:

$$\varepsilon_1 \xrightarrow[]{CF(\varepsilon_1,e_1)} e_1 \xrightarrow[]{CF(e_1,h)} h$$

$$CF(h, \varepsilon_1) = CF(h, e_1) \cdot \max(0, CF(\varepsilon_1, e_1))$$

### Combinaciones lógicas

Las evidencias se pueden agrupar mediante los operadores lógicos *AND* y *OR*:

$$CF(e_1 \land e_2) = \min(CF(e_1), CF(e_2))$$

$$CF(e_1 \lor e_2) = \max(CF(e_1), CF(e_2))$$

### Definición de reglas

Dada la regla:

$$\text{SI } e_1 \land (e_2 \lor e_3) \text{ ENTONCES } h \text{ con } CF_{regla} = x$$

Y evidencias imprecisas:

$$E = {\varepsilon_1, \varepsilon_2, \varepsilon_3}$$

Se puede utilizar el modelo de factores de certidumbre para modelarla:

$$
CF(h, E) = CF_{regla} \cdot \max\left(0,
\min\left(CF(e_1, \varepsilon_1),
\max(CF(e_2, \varepsilon_2), CF(e_3, \varepsilon_3))\right)
\right)
$$

#### Ejercicio 3

Implementa la regla

$$\text{SI } e_1 \land (e_2 \lor e_3) \text{ ENTONCES } h \text{ con } CF_{regla} = x$$

In [ ]:
### YOUR CODE HERE